In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# V1

In [ ]:
pip install git+https://github.com/csebuetnlp/normalizer

# Random Oversampling

In [ ]:
!pip install --upgrade scikit-learn imbalanced-learn

# Adasyn

# Import all required libraries at the top
from sklearn.metrics import average_precision_score, classification_report
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import random
from torch.utils.data import DataLoader
from transformers import (
    AutoConfig, 
    AutoModelForSequenceClassification, 
    AutoTokenizer,
    get_linear_schedule_with_warmup
)
from sklearn.metrics import roc_auc_score, accuracy_score, recall_score, precision_score, f1_score
from tqdm import tqdm
from normalizer import normalize

# Add these imports for oversampling techniques
from imblearn.over_sampling import RandomOverSampler, SMOTE, ADASYN
from collections import Counter

def apply_oversampling(texts, labels, ros=False, smote=False, adasyn=False, 
                      sampling_ratio=1.0, random_state=42):
    """
    Apply oversampling techniques to balance the dataset
    
    Args:
        texts: List of text data
        labels: List of corresponding labels
        ros: Boolean, use Random Over Sampling
        smote: Boolean, use SMOTE
        adasyn: Boolean, use ADASYN
        sampling_ratio: Float, controls how much to oversample minority class
                       1.0 = complete balancing (50-50)
                       0.5 = minority class becomes 50% of majority class size
                       0.2 = minority class becomes 20% of majority class size
        random_state: Random state for reproducibility
    
    Returns:
        Tuple of (resampled_texts, resampled_labels)
    """
    if not any([ros, smote, adasyn]):
        print("No oversampling technique selected. Using original data.")
        return texts, labels
    
    print(f"Original class distribution: {Counter(labels)}")
    print(f"Original class balance ratio: {min(Counter(labels).values()) / max(Counter(labels).values()):.3f}")
    
    # Calculate target sampling strategy based on sampling_ratio
    label_counts = Counter(labels)
    majority_class = max(label_counts, key=label_counts.get)
    minority_class = min(label_counts, key=label_counts.get)
    
    majority_count = label_counts[majority_class]
    target_minority_count = int(majority_count * sampling_ratio)
    
    # Create sampling strategy dictionary
    sampling_strategy = {minority_class: target_minority_count}
    
    print(f"Sampling ratio: {sampling_ratio}")
    print(f"Target minority class count: {target_minority_count}")
    
    # Convert texts to numpy array for sklearn compatibility
    X = np.array(texts).reshape(-1, 1)
    y = np.array(labels)
    
    if ros:
        print("Applying Random Over Sampling...")
        sampler = RandomOverSampler(sampling_strategy=sampling_strategy, random_state=random_state)
        X_resampled, y_resampled = sampler.fit_resample(X, y)
        resampled_texts = X_resampled.flatten().tolist()
        
    elif smote:
        print("Applying SMOTE...")
        from sklearn.neighbors import NearestNeighbors
        
        text_lengths = np.array([len(text) for text in texts])
        word_counts = np.array([len(text.split()) for text in texts])
        X_features = np.column_stack([text_lengths, word_counts])
        
        sampler = SMOTE(sampling_strategy=sampling_strategy, random_state=random_state)
        X_resampled, y_resampled = sampler.fit_resample(X_features, y)
        
        # Use sklearn's optimized nearest neighbors
        nn = NearestNeighbors(n_neighbors=1, metric='manhattan')
        nn.fit(X_features)
        _, original_indices = nn.kneighbors(X_resampled)
        original_indices = original_indices.flatten()
        
        resampled_texts = [texts[idx] for idx in original_indices]

    elif adasyn:
        print("Applying ADASYN...")
        
        # Pre-compute features once to avoid repeated calculations
        print("Computing text features...")
        text_lengths = np.array([len(text) for text in texts], dtype=np.int32)
        word_counts = np.array([len(text.split()) for text in texts], dtype=np.int32)
        X_features = np.column_stack((text_lengths, word_counts))
        
        # Apply ADASYN
        sampler = ADASYN(sampling_strategy=sampling_strategy, random_state=random_state)
        X_resampled, y_resampled = sampler.fit_resample(X_features, y)
        
        print(f"Finding closest matches for {len(X_resampled)} samples...")
        
        # Memory-efficient batch processing for finding closest matches
        batch_size = min(1000, len(X_resampled))  # Process in smaller batches
        original_indices = []
        
        # Pre-compute original features for faster distance calculation
        orig_features = X_features  # Already computed above
        
        for batch_start in range(0, len(X_resampled), batch_size):
            batch_end = min(batch_start + batch_size, len(X_resampled))
            batch_features = X_resampled[batch_start:batch_end]
            
            # Vectorized distance calculation using broadcasting
            # Shape: (batch_size, 1, 2) - (1, n_original, 2) = (batch_size, n_original, 2)
            distances = np.abs(batch_features[:, None, :] - orig_features[None, :, :])
            # Sum across feature dimensions (axis=2) to get total distance
            total_distances = np.sum(distances, axis=2)
            
            # Find closest indices for this batch
            batch_indices = np.argmin(total_distances, axis=1)
            original_indices.extend(batch_indices)
            
            # Clear memory
            del distances, total_distances, batch_features
        
        # Create resampled texts using list comprehension (faster than loop)
        resampled_texts = [texts[idx] for idx in original_indices]
        
        # Clean up large arrays
        del X_features, orig_features, X_resampled
        
        print(f"Resampled class distribution: {Counter(y_resampled)}")
        return resampled_texts, y_resampled.tolist()
        
    '''elif adasyn:
        print("Applying ADASYN...")
        # Similar approach as SMOTE for text data
        X_features = np.array([[len(text), len(text.split())] for text in texts])
        sampler = ADASYN(sampling_strategy=sampling_strategy, random_state=random_state)
        X_resampled, y_resampled = sampler.fit_resample(X_features, y)
        
        original_indices = []
        for i, (length, word_count) in enumerate(X_resampled):
            distances = []
            for j, text in enumerate(texts):
                dist = abs(len(text) - length) + abs(len(text.split()) - word_count)
                distances.append((dist, j))
            closest_idx = min(distances)[1]
            original_indices.append(closest_idx)
        
        resampled_texts = [texts[idx] for idx in original_indices]'''
    
    print(f"Resampled class distribution: {Counter(y_resampled)}")
    return resampled_texts, y_resampled.tolist()

def run(seed, model_name, lr, train_loc, val_loc, test_loc, 
        ros=False, smote=False, adasyn=False, sampling_ratio=1.0):
    # Set seeds for reproducibility
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
    # Load data
    train = pd.read_csv(train_loc)
    test = pd.read_csv(test_loc)
    val = pd.read_csv(val_loc)
    
    # Hyperparameters
    batch_size = 16
    max_tokens = 512
    epochs = 200
    best_roc_auc = 0.0
    min_delta = 0.0001
    early_stopping_count = 0
    early_stopping_patience = 3
    gradient_accumulation_steps = 2
    weight_decay = 0.01
    pruning_ratio = 0.3
    dropout_prob = 0.2
    
    # Load model configuration and tokenizer
    config = AutoConfig.from_pretrained(model_name, num_labels=2)
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # Handle tokenizers without pad_token (like XLM-RoBERTa)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    # Load the pre-trained model with the specified configuration
    model = AutoModelForSequenceClassification.from_pretrained(model_name, config=config)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    class LosDataset(torch.utils.data.Dataset):
        def __init__(self, encodings, labels):
            self.encodings = encodings
            self.labels = labels
    
        def __getitem__(self, idx):
            item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
            item['labels'] = torch.tensor(self.labels[idx])
            return item
    
        def __len__(self):
            return len(self.labels)

    # Normalize text data
    train['text'] = train['text'].apply(normalize)
    val['text'] = val['text'].apply(normalize)
    test['text'] = test['text'].apply(normalize)

    # Apply oversampling to training data only
    train_texts_resampled, train_labels_resampled = apply_oversampling(
        texts=train['text'].tolist(),
        labels=train['los_label'].tolist(),
        ros=ros,
        smote=smote,
        adasyn=adasyn,
        sampling_ratio=sampling_ratio,
        random_state=seed
)

    print(f"\n--- Training Data Distribution Summary ---")
    print(f"Original training size: {len(train['los_label'])}")
    print(f"Resampled training size: {len(train_labels_resampled)}")
    print(f"Size increase: {len(train_labels_resampled) - len(train['los_label'])} samples")
    print(f"Final class balance ratio: {min(Counter(train_labels_resampled).values()) / max(Counter(train_labels_resampled).values()):.3f}")
    print(f"--- End Distribution Summary ---\n")

    # Initialize loss function (standard CrossEntropyLoss)
    criterion = nn.CrossEntropyLoss()
    criterion = criterion.to(device)

    # Tokenize data (use resampled training data)
    train_encodings = tokenizer(train_texts_resampled, truncation=True, padding=True, max_length=max_tokens)
    val_encodings = tokenizer(val['text'].tolist(), truncation=True, padding=True, max_length=max_tokens)
    test_encodings = tokenizer(test['text'].tolist(), truncation=True, padding=True, max_length=max_tokens)

    # Create datasets
    train_dataset = LosDataset(train_encodings, train_labels_resampled)
    val_dataset = LosDataset(val_encodings, val['los_label'].tolist())
    test_dataset = LosDataset(test_encodings, test['los_label'].tolist())

    # Create data loaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, pin_memory=True)

    # Setup optimizer and scheduler
    warmup_ratio = 0.1
    total_training_steps = (epochs * len(train_loader)) // gradient_accumulation_steps
    num_warmup_steps = int(warmup_ratio * total_training_steps)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=num_warmup_steps,
        num_training_steps=total_training_steps
    )

    scaler = torch.cuda.amp.GradScaler()
    
    # Training loop
    for epoch in range(epochs):
        model.train()
        train_loss = 0
    
        for step, batch in enumerate(tqdm(train_loader, desc=f"Training Epoch {epoch+1}")):
            if step % gradient_accumulation_steps == 0:
                optimizer.zero_grad()
    
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
    
            # Mixed precision training
            with torch.amp.autocast('cuda'):
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                logits = outputs.logits
                loss = criterion(logits, labels)
                loss = loss / gradient_accumulation_steps
            
            scaler.scale(loss).backward()
            train_loss += loss.item()
            
            if (step + 1) % gradient_accumulation_steps == 0 or (step + 1) == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
    
        # Average training loss
        train_loss /= len(train_loader)
    
        # Validation loop
        model.eval()
        val_loss = 0
        val_preds = []
        val_labels = []
    
        with torch.no_grad():
            for batch in tqdm(val_loader, desc="Validation"):
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels = batch["labels"].to(device)
    
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                logits = outputs.logits
                # Use the selected criterion for validation loss too
                loss = criterion(logits, labels)
    
                val_loss += loss.item()
                probabilities = F.softmax(logits, dim=1)
                val_preds.append(probabilities.cpu().numpy())
                val_labels.append(labels.cpu().numpy())
    
        # Calculate validation metrics
        # Calculate validation metrics
        val_preds = np.concatenate(val_preds)
        val_labels = np.concatenate(val_labels)
        val_loss /= len(val_loader)
        
        # Ensure probabilities sum to 1
        if not np.allclose(val_preds.sum(axis=1), 1.0, atol=1e-6):
            print("Warning: val_preds do not sum to 1. Fixing with normalization.")
            val_preds = val_preds / val_preds.sum(axis=1, keepdims=True)
        
        val_preds_class = np.argmax(val_preds, axis=1)
        accuracy = accuracy_score(val_labels, val_preds_class)
        recall = recall_score(val_labels, val_preds_class)
        precision = precision_score(val_labels, val_preds_class)
        f1 = f1_score(val_labels, val_preds_class)
        roc_auc = roc_auc_score(val_labels, val_preds[:, 1])
        pr_auc = average_precision_score(val_labels, val_preds[:, 1])  # Add this line
        
        print(f"Epoch: {epoch+1}/{epochs}, Training Loss: {train_loss:.4f}, Validation Loss: {val_loss:.4f}")
        print(f"Accuracy: {accuracy:.4f}, Recall: {recall:.4f}, Precision: {precision:.4f}, F1: {f1:.4f}, ROC AUC: {roc_auc:.4f}, PR AUC: {pr_auc:.4f}")
    
        # Early stopping logic
        if roc_auc - best_roc_auc < min_delta:
            early_stopping_count += 1
            print(f"EarlyStopping counter: {early_stopping_count} out of {early_stopping_patience}")
            if early_stopping_count >= early_stopping_patience:
                print("Early stopping")
                break
        else:
            best_roc_auc = roc_auc
            early_stopping_count = 0

    # Test evaluation
    model.eval()
    test_loss = 0
    test_preds = []
    test_labels = []
    
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Testing"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
    
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
    
            # Use the selected criterion for test loss
            loss = criterion(logits, labels)
            test_loss += loss.item()
    
            test_preds.append(F.softmax(logits, dim=1).cpu().numpy())
            test_labels.append(labels.cpu().numpy())
    
    # Calculate test metrics
    test_preds = np.concatenate(test_preds)
    test_labels = np.concatenate(test_labels)
    test_loss /= len(test_loader)
    
    test_preds_class = np.argmax(test_preds, axis=1)
    accuracy = accuracy_score(test_labels, test_preds_class)
    recall = recall_score(test_labels, test_preds_class)
    precision = precision_score(test_labels, test_preds_class)
    f1 = f1_score(test_labels, test_preds_class)
    roc_auc = roc_auc_score(test_labels, test_preds[:, 1])
    pr_auc = average_precision_score(test_labels, test_preds[:, 1])  # Add this line
    
    print(f"\nTest Results:")
    print(f"Test Loss: {test_loss:.4f}")
    print(f"Accuracy: {accuracy:.4f}, Recall: {recall:.4f}, Precision: {precision:.4f}, F1: {f1:.4f}")
    print(f"ROC AUC: {roc_auc:.4f}, PR AUC: {pr_auc:.4f}")  # Add PR AUC here
    
    # Add classification report
    print(f"\nClassification Report:")
    print(classification_report(test_labels, test_preds_class, target_names=['Class 0', 'Class 1']))
    
    return {
    'test_accuracy': accuracy,
    'test_recall': recall,
    'test_precision': precision,
    'test_f1': f1,
    'test_roc_auc': roc_auc,
    'test_pr_auc': pr_auc,  # Add this line
    'best_val_roc_auc': best_roc_auc
}

# Dataset configuration
Mp = {
    'google_1000': [
        4e-5,
        '/kaggle/input/mahdy-research-academy-all-datasets/Mortality prediction/Mortality prediction/Mortality prediction/1000 instances google MP/1000 instances google test.csv',
        '/kaggle/input/mahdy-research-academy-all-datasets/Mortality prediction/Mortality prediction/Mortality prediction/1000 instances google MP/1000 instances google train.csv',
        '/kaggle/input/mahdy-research-academy-all-datasets/Mortality prediction/Mortality prediction/Mortality prediction/1000 instances google MP/1000 instances google val.csv'
    ],
    'opus_1000': [
        3e-5,
        '/kaggle/input/mahdy-research-academy-all-datasets/Mortality prediction/Mortality prediction/Mortality prediction/1000 instances opus MP/1000 instances opus test.csv',
        '/kaggle/input/mahdy-research-academy-all-datasets/Mortality prediction/Mortality prediction/Mortality prediction/1000 instances opus MP/1000 instances opus train.csv',
        '/kaggle/input/mahdy-research-academy-all-datasets/Mortality prediction/Mortality prediction/Mortality prediction/1000 instances opus MP/1000 instances opus val.csv'
    ],
    'banglanmt_1000': [
        4e-5,
        '/kaggle/input/mahdy-research-academy-all-datasets/Mortality prediction/Mortality prediction/Mortality prediction/BanglaNMT 1000 instances/BanglaNMT 1000 instances/BanglaNMT 1000 instances/1000 BanglaNMT(W=800) test MP.csv',
        '/kaggle/input/mahdy-research-academy-all-datasets/Mortality prediction/Mortality prediction/Mortality prediction/BanglaNMT 1000 instances/BanglaNMT 1000 instances/BanglaNMT 1000 instances/1000 BanglaNMT(W=800) Train MP.csv',
        '/kaggle/input/mahdy-research-academy-all-datasets/Mortality prediction/Mortality prediction/Mortality prediction/BanglaNMT 1000 instances/BanglaNMT 1000 instances/BanglaNMT 1000 instances/1000 BanglaNMT(W=800) val MP.csv'
    ],
    'banglanmt_full': [
        5e-5,
        '/kaggle/input/mahdy-research-academy-all-datasets/Mortality prediction/Mortality prediction/Mortality prediction/Bangla_NMT_MP_Full/Bangla_NMT_MP_Full/BanglaNMT_MP_Test.csv',
        '/kaggle/input/mahdy-research-academy-all-datasets/Mortality prediction/Mortality prediction/Mortality prediction/Bangla_NMT_MP_Full/Bangla_NMT_MP_Full/BanglaNMT_MP_Train.csv',
        '/kaggle/input/mahdy-research-academy-all-datasets/Mortality prediction/Mortality prediction/Mortality prediction/Bangla_NMT_MP_Full/Bangla_NMT_MP_Full/BanglaNMT_MP_Val.csv'
    ],
    'chatgpt_1000': [
        5e-5,
        '/kaggle/input/mahdy-research-academy-all-datasets/Mortality prediction/Mortality prediction/Mortality prediction/Chatgpt MP 1000 instances/Chatgpt 1000 instances test.csv',
        '/kaggle/input/mahdy-research-academy-all-datasets/Mortality prediction/Mortality prediction/Mortality prediction/Chatgpt MP 1000 instances/Chatgpt 1000 instances train.csv',
        '/kaggle/input/mahdy-research-academy-all-datasets/Mortality prediction/Mortality prediction/Mortality prediction/Chatgpt MP 1000 instances/Chatgpt 1000 instances val.csv'
    ],
    'opus_full': [
        2e-5,
        '/kaggle/input/mahdy-research-academy-all-datasets/Mortality prediction/Mortality prediction/Mortality prediction/MP Opus All translated/Mortality_prediction Opus Translated Test All__.csv',
        '/kaggle/input/mahdy-research-academy-all-datasets/Mortality prediction/Mortality prediction/Mortality prediction/MP Opus All translated/Mortality_prediction Opus Translated Train All.csv',
        '/kaggle/input/mahdy-research-academy-all-datasets/Mortality prediction/Mortality prediction/Mortality prediction/MP Opus All translated/Mortality_prediction Opus Translated Val All.csv'
    ]
}



# Main execution loop
if __name__ == "__main__":
    models = [
        'FacebookAI/xlm-roberta-base',
        'google-bert/bert-base-multilingual-uncased',
        'csebuetnlp/banglishbert', 
        'sagorsarker/bangla-bert-base'
    ]

    models = [models[3]]  # Using bangla-bert-base
    seeds = [42]
    results = []

    Mp_config = Mp['banglanmt_full']
    
    for seed in seeds:
        for model_name in models:
            lr = Mp_config[0]
            test_loc = Mp_config[1]
            train_loc = Mp_config[2]
            val_loc = Mp_config[3]
            
            print(f"\n{'='*50}")
            print(f"Running: Seed={seed}, Model={model_name}")
            print(f"Learning Rate: {lr}")
            print(f"{'='*50}")
            
            try:
                # Choose your oversampling technique and ratio here:
                # Set only one technique to True, and adjust sampling_ratio as needed
                result = run(
                    seed=seed, 
                    model_name=model_name, 
                    lr=lr, 
                    train_loc=train_loc, 
                    val_loc=val_loc, 
                    test_loc=test_loc,
                    ros=False,           # Random Over Sampling
                    smote=False,        # SMOTE
                    adasyn=True,       # ADASYN
                    sampling_ratio=.5  # 1.0=complete balancing, 0.5=half balancing, etc.
                )
                result['seed'] = seed
                result['model'] = model_name
                result['dataset'] = 'banglanmt_full'
                result['learning_rate'] = lr
                results.append(result)
                
            except Exception as e:
                print(f"Error running {model_name} with seed {seed}: {str(e)}")
                continue
    
    # Save results to CSV
    if results:
        results_df = pd.DataFrame(results)
        results_df.to_csv('experiment_results.csv', index=False)
        print(f"\nExperiment completed. Results saved to experiment_results.csv")
        print(f"Total successful runs: {len(results)}")

# New

In [ ]:
# Import all required libraries at the top
from sklearn.metrics import average_precision_score, classification_report
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import random
from torch.utils.data import DataLoader
from transformers import (
    AutoConfig, 
    AutoModelForSequenceClassification, 
    AutoTokenizer,
    get_linear_schedule_with_warmup
)
from sklearn.metrics import roc_auc_score, accuracy_score, recall_score, precision_score, f1_score
from tqdm import tqdm
from normalizer import normalize
import os
import copy

# Add these imports for oversampling techniques
from imblearn.over_sampling import RandomOverSampler, SMOTE, ADASYN
from collections import Counter

def apply_oversampling(texts, labels, ros=False, smote=False, adasyn=False, 
                      sampling_ratio=1.0, random_state=42):
    """
    Apply oversampling techniques to balance the dataset
    
    Args:
        texts: List of text data
        labels: List of corresponding labels
        ros: Boolean, use Random Over Sampling
        smote: Boolean, use SMOTE
        adasyn: Boolean, use ADASYN
        sampling_ratio: Float, controls how much to oversample minority class
                       1.0 = complete balancing (50-50)
                       0.5 = minority class becomes 50% of majority class size
                       0.2 = minority class becomes 20% of majority class size
        random_state: Random state for reproducibility
    
    Returns:
        Tuple of (resampled_texts, resampled_labels)
    """
    if not any([ros, smote, adasyn]):
        print("No oversampling technique selected. Using original data.")
        return texts, labels
    
    print(f"Original class distribution: {Counter(labels)}")
    print(f"Original class balance ratio: {min(Counter(labels).values()) / max(Counter(labels).values()):.3f}")
    
    # Calculate target sampling strategy based on sampling_ratio
    label_counts = Counter(labels)
    majority_class = max(label_counts, key=label_counts.get)
    minority_class = min(label_counts, key=label_counts.get)
    
    majority_count = label_counts[majority_class]
    target_minority_count = int(majority_count * sampling_ratio)
    
    # Create sampling strategy dictionary
    sampling_strategy = {minority_class: target_minority_count}
    
    print(f"Sampling ratio: {sampling_ratio}")
    print(f"Target minority class count: {target_minority_count}")
    
    # Convert texts to numpy array for sklearn compatibility
    X = np.array(texts).reshape(-1, 1)
    y = np.array(labels)
    
    if ros:
        print("Applying Random Over Sampling...")
        sampler = RandomOverSampler(sampling_strategy=sampling_strategy, random_state=random_state)
        X_resampled, y_resampled = sampler.fit_resample(X, y)
        resampled_texts = X_resampled.flatten().tolist()
        
    elif smote:
        print("Applying SMOTE...")
        from sklearn.neighbors import NearestNeighbors
        
        text_lengths = np.array([len(text) for text in texts])
        word_counts = np.array([len(text.split()) for text in texts])
        X_features = np.column_stack([text_lengths, word_counts])
        
        sampler = SMOTE(sampling_strategy=sampling_strategy, random_state=random_state)
        X_resampled, y_resampled = sampler.fit_resample(X_features, y)
        
        # Use sklearn's optimized nearest neighbors
        nn = NearestNeighbors(n_neighbors=1, metric='manhattan')
        nn.fit(X_features)
        _, original_indices = nn.kneighbors(X_resampled)
        original_indices = original_indices.flatten()
        
        resampled_texts = [texts[idx] for idx in original_indices]
        
    elif adasyn:
        print("Applying ADASYN...")
        
        # Pre-compute features once to avoid repeated calculations
        print("Computing text features...")
        text_lengths = np.array([len(text) for text in texts], dtype=np.int32)
        word_counts = np.array([len(text.split()) for text in texts], dtype=np.int32)
        X_features = np.column_stack((text_lengths, word_counts))
        
        # Apply ADASYN
        sampler = ADASYN(sampling_strategy=sampling_strategy, random_state=random_state)
        X_resampled, y_resampled = sampler.fit_resample(X_features, y)
        
        print(f"Finding closest matches for {len(X_resampled)} samples...")
        
        # Memory-efficient batch processing for finding closest matches
        batch_size = min(1000, len(X_resampled))  # Process in smaller batches
        original_indices = []
        
        # Pre-compute original features for faster distance calculation
        orig_features = X_features  # Already computed above
        
        for batch_start in range(0, len(X_resampled), batch_size):
            batch_end = min(batch_start + batch_size, len(X_resampled))
            batch_features = X_resampled[batch_start:batch_end]
            
            # Vectorized distance calculation using broadcasting
            # Shape: (batch_size, 1, 2) - (1, n_original, 2) = (batch_size, n_original, 2)
            distances = np.abs(batch_features[:, None, :] - orig_features[None, :, :])
            # Sum across feature dimensions (axis=2) to get total distance
            total_distances = np.sum(distances, axis=2)
            
            # Find closest indices for this batch
            batch_indices = np.argmin(total_distances, axis=1)
            original_indices.extend(batch_indices)
            
            # Clear memory
            del distances, total_distances, batch_features
        
        # Create resampled texts using list comprehension (faster than loop)
        resampled_texts = [texts[idx] for idx in original_indices]
        
        # Clean up large arrays
        del X_features, orig_features, X_resampled
        
        print(f"Resampled class distribution: {Counter(y_resampled)}")
        return resampled_texts, y_resampled.tolist()

def run(seed, model_name, lr, train_loc, val_loc, test_loc, 
        ros=False, smote=False, adasyn=False, sampling_ratio=1.0):
    # Set seeds for reproducibility
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
    # Load data
    train = pd.read_csv(train_loc)
    test = pd.read_csv(test_loc)
    val = pd.read_csv(val_loc)
    
    # Hyperparameters
    batch_size = 16
    max_tokens = 512
    epochs = 200
    best_roc_auc = 0.0
    min_delta = 0.0001
    early_stopping_count = 0
    early_stopping_patience = 3
    gradient_accumulation_steps = 2
    weight_decay = 0.01
    pruning_ratio = 0.3
    dropout_prob = 0.2
    
    # Load model configuration and tokenizer
    config = AutoConfig.from_pretrained(model_name, num_labels=2)
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # Handle tokenizers without pad_token (like XLM-RoBERTa)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    # Load the pre-trained model with the specified configuration
    model = AutoModelForSequenceClassification.from_pretrained(model_name, config=config)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    class LosDataset(torch.utils.data.Dataset):
        def __init__(self, encodings, labels):
            self.encodings = encodings
            self.labels = labels
    
        def __getitem__(self, idx):
            item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
            item['labels'] = torch.tensor(self.labels[idx])
            return item
    
        def __len__(self):
            return len(self.labels)

    # Normalize text data
    train['text'] = train['text'].apply(normalize)
    val['text'] = val['text'].apply(normalize)
    test['text'] = test['text'].apply(normalize)

    # Apply oversampling to training data only
    train_texts_resampled, train_labels_resampled = apply_oversampling(
        texts=train['text'].tolist(),
        labels=train['los_label'].tolist(),
        ros=ros,
        smote=smote,
        adasyn=adasyn,
        sampling_ratio=sampling_ratio,
        random_state=seed
    )

    print(f"\n--- Training Data Distribution Summary ---")
    print(f"Original training size: {len(train['los_label'])}")
    print(f"Resampled training size: {len(train_labels_resampled)}")
    print(f"Size increase: {len(train_labels_resampled) - len(train['los_label'])} samples")
    print(f"Final class balance ratio: {min(Counter(train_labels_resampled).values()) / max(Counter(train_labels_resampled).values()):.3f}")
    print(f"--- End Distribution Summary ---\n")

    # Initialize loss function (standard CrossEntropyLoss)
    criterion = nn.CrossEntropyLoss()
    criterion = criterion.to(device)

    # Tokenize data (use resampled training data)
    train_encodings = tokenizer(train_texts_resampled, truncation=True, padding=True, max_length=max_tokens)
    val_encodings = tokenizer(val['text'].tolist(), truncation=True, padding=True, max_length=max_tokens)
    test_encodings = tokenizer(test['text'].tolist(), truncation=True, padding=True, max_length=max_tokens)

    # Create datasets
    train_dataset = LosDataset(train_encodings, train_labels_resampled)
    val_dataset = LosDataset(val_encodings, val['los_label'].tolist())
    test_dataset = LosDataset(test_encodings, test['los_label'].tolist())

    # Create data loaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, pin_memory=True)

    # Setup optimizer and scheduler
    warmup_ratio = 0.1
    total_training_steps = (epochs * len(train_loader)) // gradient_accumulation_steps
    num_warmup_steps = int(warmup_ratio * total_training_steps)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=num_warmup_steps,
        num_training_steps=total_training_steps
    )

    scaler = torch.cuda.amp.GradScaler()
    
    # Initialize best model state
    best_model_state = None
    
    # Training loop
    for epoch in range(epochs):
        model.train()
        train_loss = 0
    
        for step, batch in enumerate(tqdm(train_loader, desc=f"Training Epoch {epoch+1}")):
            if step % gradient_accumulation_steps == 0:
                optimizer.zero_grad()
    
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
    
            # Mixed precision training
            with torch.amp.autocast('cuda'):
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                logits = outputs.logits
                loss = criterion(logits, labels)
                loss = loss / gradient_accumulation_steps
            
            scaler.scale(loss).backward()
            train_loss += loss.item()
            
            if (step + 1) % gradient_accumulation_steps == 0 or (step + 1) == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
    
        # Average training loss
        train_loss /= len(train_loader)
    
        # Validation loop
        model.eval()
        val_loss = 0
        val_preds = []
        val_labels = []
    
        with torch.no_grad():
            for batch in tqdm(val_loader, desc="Validation"):
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels = batch["labels"].to(device)
    
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                logits = outputs.logits
                # Use the selected criterion for validation loss too
                loss = criterion(logits, labels)
    
                val_loss += loss.item()
                probabilities = F.softmax(logits, dim=1)
                val_preds.append(probabilities.cpu().numpy())
                val_labels.append(labels.cpu().numpy())
    
        # Calculate validation metrics
        val_preds = np.concatenate(val_preds)
        val_labels = np.concatenate(val_labels)
        val_loss /= len(val_loader)
        
        # Ensure probabilities sum to 1
        if not np.allclose(val_preds.sum(axis=1), 1.0, atol=1e-6):
            print("Warning: val_preds do not sum to 1. Fixing with normalization.")
            val_preds = val_preds / val_preds.sum(axis=1, keepdims=True)
        
        val_preds_class = np.argmax(val_preds, axis=1)
        accuracy = accuracy_score(val_labels, val_preds_class)
        recall = recall_score(val_labels, val_preds_class)
        precision = precision_score(val_labels, val_preds_class)
        f1 = f1_score(val_labels, val_preds_class)
        roc_auc = roc_auc_score(val_labels, val_preds[:, 1])
        pr_auc = average_precision_score(val_labels, val_preds[:, 1])
        
        print(f"Epoch: {epoch+1}/{epochs}, Training Loss: {train_loss:.4f}, Validation Loss: {val_loss:.4f}")
        print(f"Accuracy: {accuracy:.4f}, Recall: {recall:.4f}, Precision: {precision:.4f}, F1: {f1:.4f}, ROC AUC: {roc_auc:.4f}, PR AUC: {pr_auc:.4f}")
    
        # Save best model based on ROC AUC
        if roc_auc > best_roc_auc:
            best_roc_auc = roc_auc
            best_model_state = copy.deepcopy(model.state_dict())
            early_stopping_count = 0
            print(f"New best model saved with ROC AUC: {best_roc_auc:.4f}")
        else:
            early_stopping_count += 1
            print(f"EarlyStopping counter: {early_stopping_count} out of {early_stopping_patience}")
            if early_stopping_count >= early_stopping_patience:
                print("Early stopping")
                break

    # Load the best model for test evaluation
    print(f"\nLoading best model for test evaluation (ROC AUC: {best_roc_auc:.4f})")
    model.load_state_dict(best_model_state)
    
    # Test evaluation with best model
    model.eval()
    test_loss = 0
    test_preds = []
    test_labels = []
    
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Testing with Best Model"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
    
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
    
            # Use the selected criterion for test loss
            loss = criterion(logits, labels)
            test_loss += loss.item()
    
            test_preds.append(F.softmax(logits, dim=1).cpu().numpy())
            test_labels.append(labels.cpu().numpy())
    
    # Calculate test metrics
    test_preds = np.concatenate(test_preds)
    test_labels = np.concatenate(test_labels)
    test_loss /= len(test_loader)
    
    test_preds_class = np.argmax(test_preds, axis=1)
    accuracy = accuracy_score(test_labels, test_preds_class)
    recall = recall_score(test_labels, test_preds_class)
    precision = precision_score(test_labels, test_preds_class)
    f1 = f1_score(test_labels, test_preds_class)
    roc_auc = roc_auc_score(test_labels, test_preds[:, 1])
    pr_auc = average_precision_score(test_labels, test_preds[:, 1])
    
    print(f"\nTest Results (Best Model):")
    print(f"Best Validation ROC AUC: {best_roc_auc:.4f}")
    print(f"Test Loss: {test_loss:.4f}")
    print(f"Accuracy: {accuracy:.4f}, Recall: {recall:.4f}, Precision: {precision:.4f}, F1: {f1:.4f}")
    print(f"ROC AUC: {roc_auc:.4f}, PR AUC: {pr_auc:.4f}")
    
    # Add classification report
    print(f"\nClassification Report:")
    print(classification_report(test_labels, test_preds_class, target_names=['Class 0', 'Class 1']))
    
    return {
        'test_accuracy': accuracy,
        'test_recall': recall,
        'test_precision': precision,
        'test_f1': f1,
        'test_roc_auc': roc_auc,
        'test_pr_auc': pr_auc,
        'best_val_roc_auc': best_roc_auc
    }

# Dataset configuration
Mp = {
    'google_1000': [
        4e-5,
        '/kaggle/input/mahdy-research-academy-all-datasets/Mortality prediction/Mortality prediction/Mortality prediction/1000 instances google MP/1000 instances google test.csv',
        '/kaggle/input/mahdy-research-academy-all-datasets/Mortality prediction/Mortality prediction/Mortality prediction/1000 instances google MP/1000 instances google train.csv',
        '/kaggle/input/mahdy-research-academy-all-datasets/Mortality prediction/Mortality prediction/Mortality prediction/1000 instances google MP/1000 instances google val.csv'
    ],
    'opus_1000': [
        3e-5,
        '/kaggle/input/mahdy-research-academy-all-datasets/Mortality prediction/Mortality prediction/Mortality prediction/1000 instances opus MP/1000 instances opus test.csv',
        '/kaggle/input/mahdy-research-academy-all-datasets/Mortality prediction/Mortality prediction/Mortality prediction/1000 instances opus MP/1000 instances opus train.csv',
        '/kaggle/input/mahdy-research-academy-all-datasets/Mortality prediction/Mortality prediction/Mortality prediction/1000 instances opus MP/1000 instances opus val.csv'
    ],
    'banglanmt_1000': [
        4e-5,
        '/kaggle/input/mahdy-research-academy-all-datasets/Mortality prediction/Mortality prediction/Mortality prediction/BanglaNMT 1000 instances/BanglaNMT 1000 instances/BanglaNMT 1000 instances/1000 BanglaNMT(W=800) test MP.csv',
        '/kaggle/input/mahdy-research-academy-all-datasets/Mortality prediction/Mortality prediction/Mortality prediction/BanglaNMT 1000 instances/BanglaNMT 1000 instances/BanglaNMT 1000 instances/1000 BanglaNMT(W=800) Train MP.csv',
        '/kaggle/input/mahdy-research-academy-all-datasets/Mortality prediction/Mortality prediction/Mortality prediction/BanglaNMT 1000 instances/BanglaNMT 1000 instances/BanglaNMT 1000 instances/1000 BanglaNMT(W=800) val MP.csv'
    ],
    'banglanmt_full': [
        5e-5,
        '/kaggle/input/mahdy-research-academy-all-datasets/Mortality prediction/Mortality prediction/Mortality prediction/Bangla_NMT_MP_Full/Bangla_NMT_MP_Full/BanglaNMT_MP_Test.csv',
        '/kaggle/input/mahdy-research-academy-all-datasets/Mortality prediction/Mortality prediction/Mortality prediction/Bangla_NMT_MP_Full/Bangla_NMT_MP_Full/BanglaNMT_MP_Train.csv',
        '/kaggle/input/mahdy-research-academy-all-datasets/Mortality prediction/Mortality prediction/Mortality prediction/Bangla_NMT_MP_Full/Bangla_NMT_MP_Full/BanglaNMT_MP_Val.csv'
    ],
    'chatgpt_1000': [
        5e-5,
        '/kaggle/input/mahdy-research-academy-all-datasets/Mortality prediction/Mortality prediction/Mortality prediction/Chatgpt MP 1000 instances/Chatgpt 1000 instances test.csv',
        '/kaggle/input/mahdy-research-academy-all-datasets/Mortality prediction/Mortality prediction/Mortality prediction/Chatgpt MP 1000 instances/Chatgpt 1000 instances train.csv',
        '/kaggle/input/mahdy-research-academy-all-datasets/Mortality prediction/Mortality prediction/Mortality prediction/Chatgpt MP 1000 instances/Chatgpt 1000 instances val.csv'
    ],
    'opus_full': [
        2e-5,
        '/kaggle/input/mahdy-research-academy-all-datasets/Mortality prediction/Mortality prediction/Mortality prediction/MP Opus All translated/Mortality_prediction Opus Translated Test All__.csv',
        '/kaggle/input/mahdy-research-academy-all-datasets/Mortality prediction/Mortality prediction/Mortality prediction/MP Opus All translated/Mortality_prediction Opus Translated Train All.csv',
        '/kaggle/input/mahdy-research-academy-all-datasets/Mortality prediction/Mortality prediction/Mortality prediction/MP Opus All translated/Mortality_prediction Opus Translated Val All.csv'
    ]
}

# Main execution loop
if __name__ == "__main__":
    models = [
        'FacebookAI/xlm-roberta-base',
        'google-bert/bert-base-multilingual-uncased',
        'csebuetnlp/banglishbert', 
        'sagorsarker/bangla-bert-base'
    ]

    models = [models[3]]  # Using bangla-bert-base
    seeds = [42]
    results = []

    Mp_config = Mp['banglanmt_full']
    
    for seed in seeds:
        for model_name in models:
            lr = Mp_config[0]
            test_loc = Mp_config[1]
            train_loc = Mp_config[2]
            val_loc = Mp_config[3]
            
            print(f"\n{'='*50}")
            print(f"Running: Seed={seed}, Model={model_name}")
            print(f"Learning Rate: {lr}")
            print(f"{'='*50}")
            
            try:
                # Choose your oversampling technique and ratio here:
                # Set only one technique to True, and adjust sampling_ratio as needed
                result = run(
                    seed=seed, 
                    model_name=model_name, 
                    lr=lr, 
                    train_loc=train_loc, 
                    val_loc=val_loc, 
                    test_loc=test_loc,
                    ros=False,           # Random Over Sampling
                    smote=False,        # SMOTE
                    adasyn=True,       # ADASYN
                    sampling_ratio=.5  # 1.0=complete balancing, 0.5=half balancing, etc.
                )
                result['seed'] = seed
                result['model'] = model_name
                result['dataset'] = 'banglanmt_full'
                result['learning_rate'] = lr
                results.append(result)
                
            except Exception as e:
                print(f"Error running {model_name} with seed {seed}: {str(e)}")
                continue
    
    # Save results to CSV
    if results:
        results_df = pd.DataFrame(results)
        results_df.to_csv('experiment_results.csv', index=False)
        print(f"\nExperiment completed. Results saved to experiment_results.csv")
        print(f"Total successful runs: {len(results)}")